# Explicit Optional-Argument Presence

## tl;dr

Fresh cohort: baseline: exact 32/56, default-equivalent 35/56; candidate_v1: exact 50/56, default-equivalent 50/56; candidate_v3: exact 50/56, default-equivalent 50/56.

The failed prompt-only v2 experiment remains part of the evidence. No default or Gate promotion.

## Context & Methods

The known high-priority omission is a development regression. V2 changes the optional-field prompt only. V3 requires every optional field in a separate wire schema, using model-generated null to mean omit. Supplied values are copied unchanged.

### Key Assumptions

Four development requests and fourteen locally authored fresh requests, two catalog orders, two seeds, and three variants per run. The fresh cohort compares baseline, v1 and v3 only, after v2 failed on development. Cases are not independently reviewed. Trials/orders repeat the same requests. No production semantic guarantee or latency SLA is claimed.

## Data

### 1. Verify Inputs and Recompute

The old v2 runner is preserved as a hash-addressed source snapshot. The audit reads source bytes but never executes snapshots. Model inference is not repeated by this notebook.

In [1]:
import hashlib, json, sys
from pathlib import Path
root = Path.cwd()
while not (root / 'reference_workload/runtime_matrix.json').exists():
    assert root != root.parent
    root = root.parent
paths = {'development_v2': 'artifacts/reference-workload/tool-presence-development-v2.json', 'development_v3': 'artifacts/reference-workload/tool-presence-development-v3.json', 'fresh_v3': 'artifacts/reference-workload/tool-presence-fresh-v3.json'}
hashes = {'development_v2': 'a314ae6f3d8061e3f033356a0aa5e7a10ace0d2ea4148a603bdea5c668983f2d', 'development_v3': 'ae831f9c65194d7cbe758ecc2614c19266d02f430943dcd01bf5af0142d752ac', 'fresh_v3': 'c7d745ad01978de0ce70d8f88b20e5a383e599bfa0367a04fef291943214241a'}
helpers = {'review_tool_presence.py': 'b6a623c807df92d09acfbe4dbaf7d7b3dcda208dbb2219bbe6dd12668bb2d843', 'review_tool_default_semantics.py': '848f38fc511ff694b5cce7d06f54355764b03cc916709781e7368a1809aaf75a'}
for label, path in paths.items():
    assert hashlib.sha256((root / path).read_bytes()).hexdigest() == hashes[label]
for name, expected in helpers.items():
    assert hashlib.sha256((root / 'tools' / name).read_bytes()).hexdigest() == expected
sys.path.insert(0, str(root / 'tools'))
from review_tool_presence import audit
result = audit(root)
print(result['assessment'])

share_with_caveats_local_diagnostic_only


## Results

### 2. Development and Fresh Results

Exact and default-equivalent criteria are unchanged. Token totals require complete captured HTTP usage. Median wall time is descriptive.

In [2]:
from IPython.display import Markdown, display
headers = ['cohort', 'variant', 'n', 'selection', 'exact', 'equivalent', 'total_tokens', 'median_ms']
table = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for experiment in result['experiments']:
    for row in experiment['summary']:
        values = [experiment['label'], row['variant'], row['n'], row['selection'], row['exact'], row['equivalent'], row['total_tokens'], round(row['median_latency_ms'], 1)]
        table.append('| ' + ' | '.join(map(str, values)) + ' |')
display(Markdown('\n'.join(table)))

| cohort | variant | n | selection | exact | equivalent | total_tokens | median_ms |
| --- | --- | --- | --- | --- | --- | --- | --- |
| development_v2 | baseline | 16 | 16 | 14 | 16 | 14385 | 847.3 |
| development_v2 | candidate_v1 | 16 | 16 | 12 | 12 | 18844 | 1320.9 |
| development_v2 | candidate_v2 | 16 | 16 | 12 | 12 | 21200 | 1317.5 |
| development_v3 | baseline | 16 | 16 | 14 | 16 | 14385 | 846.6 |
| development_v3 | candidate_v1 | 16 | 16 | 12 | 12 | 18844 | 1302.5 |
| development_v3 | candidate_v3 | 16 | 16 | 16 | 16 | 20228 | 1336.8 |
| fresh_v3 | baseline | 56 | 54 | 32 | 35 | 50804 | 865.2 |
| fresh_v3 | candidate_v1 | 56 | 54 | 50 | 50 | 67356 | 1384.7 |
| fresh_v3 | candidate_v3 | 56 | 54 | 50 | 50 | 70062 | 1386.8 |

### 3. Priority-Specific Checks

Fresh priority rows each represent two authored requests in two orders and two seeds, not eight independent requests.

In [3]:
fresh = next(e for e in result['experiments'] if e['label'] == 'fresh_v3')
print(json.dumps([r for r in fresh['strata'] if r['stratum'].startswith('ticket_')], indent=2))

[
  {
    "variant": "baseline",
    "stratum": "ticket_explicit_high",
    "n": 8,
    "exact": 6,
    "equivalent": 6
  },
  {
    "variant": "candidate_v1",
    "stratum": "ticket_explicit_high",
    "n": 8,
    "exact": 8,
    "equivalent": 8
  },
  {
    "variant": "candidate_v3",
    "stratum": "ticket_explicit_high",
    "n": 8,
    "exact": 8,
    "equivalent": 8
  },
  {
    "variant": "candidate_v3",
    "stratum": "ticket_explicit_low",
    "n": 8,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "baseline",
    "stratum": "ticket_explicit_low",
    "n": 8,
    "exact": 2,
    "equivalent": 2
  },
  {
    "variant": "candidate_v1",
    "stratum": "ticket_explicit_low",
    "n": 8,
    "exact": 4,
    "equivalent": 4
  },
  {
    "variant": "candidate_v1",
    "stratum": "ticket_explicit_normal",
    "n": 8,
    "exact": 8,
    "equivalent": 8
  },
  {
    "variant": "candidate_v3",
    "stratum": "ticket_explicit_normal",
    "n": 8,
    "exact": 8,
    "equivalen

### 4. Protocol Fidelity and Remaining Errors

Null omissions are model decisions. Exact values of all included fields must match the original wire response.

In [4]:
for experiment in result['experiments']:
    print(experiment['label'], {k: experiment[k] for k in (
        'observations', 'http_calls', 'replay_traces', 'required_only_stage2_body_matches',
        'model_null_omission_count', 'finish_reasons')})
print('Fresh non-equivalent observations:', len(fresh['non_equivalent_rows']))

development_v2 {'observations': 48, 'http_calls': 80, 'replay_traces': 144, 'required_only_stage2_body_matches': 0, 'model_null_omission_count': {}, 'finish_reasons': {'stop': 80}}
development_v3 {'observations': 48, 'http_calls': 80, 'replay_traces': 144, 'required_only_stage2_body_matches': 0, 'model_null_omission_count': {'candidate_v3': 4}, 'finish_reasons': {'stop': 80}}
fresh_v3 {'observations': 168, 'http_calls': 280, 'replay_traces': 504, 'required_only_stage2_body_matches': 24, 'model_null_omission_count': {'candidate_v3': 12}, 'finish_reasons': {'stop': 280}}
Fresh non-equivalent observations: 33


## Takeaways

Keep development and fresh conclusions separate. Null improves the explicitness of a model's decision but does not make that decision infallible. A wrong null can still lose a user requirement. Real Tool content, authorization, independent review and runtime intent validation remain outside this diagnostic. Historical evidence and strict scores are unchanged.

In [5]:
target = root / 'docs/reports/2026-09-09_tool_presence' / 'audit.json'
target.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Audit saved')

Audit saved
